# Filtered GT6 Hard-Sample Removal Retraining Experiment

This notebook is a **diagnostic experiment**.

It removes samples whose `wrong_multimodal_run_count > 6` from the prepared splits, then retrains the main model families to compare results before and after removing hard/ambiguous examples.

Important: because the removed samples are identified using previous test-set errors, this is **not** a fair headline benchmark. Use it as an error-analysis ablation: it tells us how much performance changes when a small hard subset is excluded.


## 1. Configuration

In [1]:
from pathlib import Path
import json
import re
import random
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 140)
pd.set_option('display.max_columns', 60)

SEED = 42
FILTER_THRESHOLD = 6
FILTER_COLUMN = 'wrong_multimodal_run_count'
FILTER_RULE = '>'  # your requested rule: delete samples wrong in more than 6 multimodal model runs

DATA_SPLIT_DIR = Path('data_splits')
FINAL_ARTIFACT_DIR = Path('final_artifacts')
EXPERIMENT_DIR = Path('filtered_gt6_retrain_artifacts')
FILTERED_SPLIT_DIR = DATA_SPLIT_DIR / 'filtered_gt6_multimodal_wrong_removed'

for p in [FINAL_ARTIFACT_DIR, EXPERIMENT_DIR, FILTERED_SPLIT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

LABEL_TO_ID = {'real': 0, 'fake': 1}
ID_TO_LABEL = {0: 'real', 1: 'fake'}

RUN_TEXT_TFIDF = True
RUN_DEEP_CNN_MODELS = True
RUN_FROZEN_RESNET50 = True
RUN_TEXT_BERT = True
RUN_RICH_CACHED_MLP = True
RUN_VOTING_ENSEMBLE = True

# Set lower for debugging; increase for the overnight/full run.
RETRAIN_SEEDS = [42, 123, 2026]
MAX_EPOCHS = 25
PATIENCE = 4
MIN_DELTA = 1e-4
BATCH_SIZE = 32
TEXT_MAX_FEATURES = 5000

print('Filtered retrain experiment directory:', EXPERIMENT_DIR)
print('Filtered split directory:', FILTERED_SPLIT_DIR)

Filtered retrain experiment directory: filtered_gt6_retrain_artifacts
Filtered split directory: data_splits\filtered_gt6_multimodal_wrong_removed


## 2. Build Filtered Splits

In [2]:
def load_final_split(split):
    candidates = [
        DATA_SPLIT_DIR / f'{split}_valid_images_only_image_deleaked_with_content.csv',
        DATA_SPLIT_DIR / f'{split}_valid_images_only_image_deleaked.csv',
        DATA_SPLIT_DIR / f'{split}_valid_images_only.csv',
        DATA_SPLIT_DIR / f'{split}_multimodal.csv',
        DATA_SPLIT_DIR / f'{split}.csv',
    ]
    for path in candidates:
        if path.exists():
            df = pd.read_csv(path)
            df['split'] = split
            if 'label_id' not in df.columns:
                df['label_id'] = df['label'].map(LABEL_TO_ID).astype(int)
            if 'valid_image_path' not in df.columns:
                img_col = 'image_path_primary' if 'image_path_primary' in df.columns else 'image_path'
                df['valid_image_path'] = df[img_col].fillna('').astype(str).map(lambda x: bool(x) and Path(x).exists()) if img_col in df.columns else False
            return df, path
    raise FileNotFoundError(f'No split file found for {split}. Checked: {candidates}')

hard_path = FINAL_ARTIFACT_DIR / 'final_multimodal_most_frequent_wrong_samples.csv'
if not hard_path.exists():
    raise FileNotFoundError('Run Notebook 3 voting/multimodal failure analysis first. Missing final_multimodal_most_frequent_wrong_samples.csv')

hard_df = pd.read_csv(hard_path)
if FILTER_RULE == '>':
    hard_ids = set(hard_df.loc[hard_df[FILTER_COLUMN] > FILTER_THRESHOLD, 'id'].astype(str))
else:
    hard_ids = set(hard_df.loc[hard_df[FILTER_COLUMN] >= FILTER_THRESHOLD, 'id'].astype(str))

print(f'Hard samples to remove where {FILTER_COLUMN} {FILTER_RULE} {FILTER_THRESHOLD}: {len(hard_ids)}')

split_rows = []
filtered_splits = {}
for split in ['train', 'val', 'test']:
    df, path = load_final_split(split)
    before = len(df)
    filtered = df[~df['id'].astype(str).isin(hard_ids)].reset_index(drop=True)
    filtered_splits[split] = filtered
    out_path = FILTERED_SPLIT_DIR / f'{split}.csv'
    filtered.to_csv(out_path, index=False)
    split_rows.append({
        'split': split,
        'loaded_from': str(path),
        'output_file': str(out_path),
        'rows_before': before,
        'rows_after': len(filtered),
        'rows_removed': before - len(filtered),
        'fake_after': int((filtered['label_id'] == 1).sum()),
        'real_after': int((filtered['label_id'] == 0).sum()),
    })

filter_audit = pd.DataFrame(split_rows)
filter_audit.to_csv(EXPERIMENT_DIR / 'filtered_gt6_split_audit.csv', index=False)
hard_df[hard_df['id'].astype(str).isin(hard_ids)].to_csv(EXPERIMENT_DIR / 'removed_gt6_hard_samples.csv', index=False)
display(filter_audit)

train_df = filtered_splits['train']
val_df = filtered_splits['val']
test_df = filtered_splits['test']

print('Filtered split label balance:')
display(pd.concat([
    train_df.assign(split='train'),
    val_df.assign(split='val'),
    test_df.assign(split='test'),
]).groupby(['split', 'label']).size().unstack(fill_value=0))

Hard samples to remove where wrong_multimodal_run_count > 6: 112


,split,loaded_from,output_file,rows_before,rows_after,rows_removed,fake_after,real_after
0,train,data_splits\train_valid_images_only_image_deleaked_with_content.csv,data_splits\filtered_gt6_multimodal_wrong_removed\train.csv,4829,4829,0,2320,2509
1,val,data_splits\val_valid_images_only_image_deleaked_with_content.csv,data_splits\filtered_gt6_multimodal_wrong_removed\val.csv,541,541,0,250,291
2,test,data_splits\test_valid_images_only_image_deleaked_with_content.csv,data_splits\filtered_gt6_multimodal_wrong_removed\test.csv,518,406,112,176,230


Filtered split label balance:


label,fake,real
split,,
test,176,230
train,2320,2509
val,250,291


## 3. Shared Metric Utilities

In [3]:
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, roc_auc_score, average_precision_score, confusion_matrix

def get_text_column(df):
    for col in ['text_for_model', 'content', 'title']:
        if col in df.columns:
            return col
    raise ValueError('No usable text column found.')

TEXT_COL = get_text_column(train_df)
print('Text column:', TEXT_COL)

def compute_metrics(y_true, y_pred, prob_fake=None, model='model'):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    p, r, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1], zero_division=0)
    row = {
        'model': model,
        'n_samples': int(len(y_true)),
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'real_precision': p[0],
        'real_recall': r[0],
        'real_f1': f1[0],
        'fake_precision': p[1],
        'fake_recall': r[1],
        'fake_f1': f1[1],
    }
    if prob_fake is not None:
        prob = pd.to_numeric(pd.Series(prob_fake), errors='coerce')
        if prob.notna().all() and len(np.unique(y_true)) == 2:
            row['roc_auc'] = roc_auc_score(y_true, prob)
            row['pr_auc'] = average_precision_score(y_true, prob)
        else:
            row['roc_auc'] = np.nan
            row['pr_auc'] = np.nan
    else:
        row['roc_auc'] = np.nan
        row['pr_auc'] = np.nan
    return row

def save_prediction_file(df, pred_id, prob_fake, model_name):
    out = df[['id', 'source', 'title', 'label', 'label_id', 'valid_image_path']].copy()
    if 'image_path_primary' in df.columns:
        out['image_path_primary'] = df['image_path_primary']
    out['pred_id'] = np.asarray(pred_id, dtype=int)
    out['pred_label'] = [ID_TO_LABEL[int(x)] for x in pred_id]
    out['prob_fake'] = prob_fake
    out['correct'] = out['pred_id'].eq(out['label_id'].astype(int))
    safe = re.sub(r'[^A-Za-z0-9_.+-]+', '_', model_name).strip('_')
    path = EXPERIMENT_DIR / f'{safe}_test_predictions.csv'
    out.to_csv(path, index=False)
    return out, path

Text column: text_for_model


## 4. Text Baseline Retraining

In [4]:
result_rows = []
prediction_frames = []

if RUN_TEXT_TFIDF:
    from sklearn.pipeline import Pipeline
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.dummy import DummyClassifier

    X_train = train_df[TEXT_COL].fillna('').astype(str)
    X_val = val_df[TEXT_COL].fillna('').astype(str)
    X_test = test_df[TEXT_COL].fillna('').astype(str)
    y_train = train_df['label_id'].astype(int).values
    y_test = test_df['label_id'].astype(int).values

    majority = DummyClassifier(strategy='most_frequent')
    majority.fit(X_train, y_train)
    maj_pred = majority.predict(X_test)
    maj_prob = np.full(len(y_test), float(np.mean(y_train == 1)))
    result_rows.append(compute_metrics(y_test, maj_pred, maj_prob, 'Filtered GT6 Majority baseline'))
    pred_df, pred_path = save_prediction_file(test_df, maj_pred, maj_prob, 'filtered_gt6_majority')
    prediction_frames.append(pred_df.assign(model_name='filtered_gt6_majority', model_family='majority'))

    tfidf_lr = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=TEXT_MAX_FEATURES, ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ('clf', LogisticRegression(max_iter=1500, class_weight='balanced', solver='liblinear', random_state=SEED)),
    ])
    tfidf_lr.fit(X_train, y_train)
    text_pred = tfidf_lr.predict(X_test)
    text_prob = tfidf_lr.predict_proba(X_test)[:, 1]
    result_rows.append(compute_metrics(y_test, text_pred, text_prob, 'Filtered GT6 TF-IDF Logistic Regression'))
    pred_df, pred_path = save_prediction_file(test_df, text_pred, text_prob, 'filtered_gt6_tfidf_logreg')
    prediction_frames.append(pred_df.assign(model_name='filtered_gt6_tfidf_logreg', model_family='text_tfidf'))

    pd.DataFrame(result_rows).to_csv(EXPERIMENT_DIR / 'filtered_gt6_text_baseline_metrics.csv', index=False)
    display(pd.DataFrame(result_rows))

,model,n_samples,accuracy,macro_f1,real_precision,real_recall,real_f1,fake_precision,fake_recall,fake_f1,roc_auc,pr_auc
0,Filtered GT6 Majority baseline,406,0.566502,0.361635,0.566502,1.000000,0.723270,0.000000,0.000000,0.000000,0.500000,0.433498
1,Filtered GT6 TF-IDF Logistic Regression,406,0.864532,0.860740,0.860082,0.908696,0.883721,0.871166,0.806818,0.837758,0.928236,0.924338


## 5. Deep Multimodal Retraining: Image-Only, Concat, Consistency, Frozen ResNet50

In [ ]:
if RUN_DEEP_CNN_MODELS:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import Dataset, DataLoader
    from PIL import Image
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.preprocessing import StandardScaler
    from scipy import sparse
    try:
        from torchvision import transforms, models
        TORCHVISION_OK = True
    except Exception as exc:
        TORCHVISION_OK = False
        print('Torchvision unavailable; deep image models will be skipped:', repr(exc))

    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Device:', DEVICE)

    def set_seed(seed):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    vectorizer = TfidfVectorizer(max_features=TEXT_MAX_FEATURES, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    X_train_text = vectorizer.fit_transform(train_df[TEXT_COL].fillna('').astype(str))
    X_val_text = vectorizer.transform(val_df[TEXT_COL].fillna('').astype(str))
    X_test_text = vectorizer.transform(test_df[TEXT_COL].fillna('').astype(str))

    scaler = StandardScaler(with_mean=False)
    X_train_text = scaler.fit_transform(X_train_text).astype(np.float32)
    X_val_text = scaler.transform(X_val_text).astype(np.float32)
    X_test_text = scaler.transform(X_test_text).astype(np.float32)

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]) if TORCHVISION_OK else None

    class FilteredMultimodalDataset(Dataset):
        def __init__(self, df, text_matrix, transform=None):
            self.df = df.reset_index(drop=True)
            self.text_matrix = text_matrix
            self.transform = transform
            self.image_col = 'image_path_primary' if 'image_path_primary' in self.df.columns else 'image_path'
        def __len__(self):
            return len(self.df)
        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            label = int(row['label_id'])
            text_vec = self.text_matrix[idx]
            if sparse.issparse(text_vec):
                text_vec = text_vec.toarray().ravel()
            text_vec = torch.tensor(text_vec, dtype=torch.float32)
            path = str(row.get(self.image_col, ''))
            try:
                img = Image.open(path).convert('RGB')
                img = self.transform(img) if self.transform else torch.zeros(3, 224, 224)
            except Exception:
                img = torch.zeros(3, 224, 224)
            return text_vec, img, torch.tensor(label, dtype=torch.long)

    class SmallImageEncoder(nn.Module):
        def __init__(self, out_dim=128):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
                nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
                nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1)),
            )
            self.proj = nn.Sequential(nn.Flatten(), nn.Dropout(0.35), nn.Linear(128, out_dim), nn.ReLU())
        def forward(self, x):
            return self.proj(self.net(x))

    class ImageOnlyNet(nn.Module):
        def __init__(self, hidden=128):
            super().__init__()
            self.image = SmallImageEncoder(hidden)
            self.head = nn.Sequential(nn.Dropout(0.45), nn.Linear(hidden, 2))
        def forward(self, text, image):
            return self.head(self.image(image))

    class ConcatFusionNet(nn.Module):
        def __init__(self, text_dim, hidden=128):
            super().__init__()
            self.image = SmallImageEncoder(hidden)
            self.text = nn.Sequential(nn.Linear(text_dim, hidden), nn.ReLU(), nn.Dropout(0.35))
            self.head = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.ReLU(), nn.Dropout(0.45), nn.Linear(hidden, 2))
        def forward(self, text, image):
            return self.head(torch.cat([self.text(text), self.image(image)], dim=1))

    class ConsistencyFusionNet(nn.Module):
        def __init__(self, text_dim, hidden=128):
            super().__init__()
            self.image = SmallImageEncoder(hidden)
            self.text = nn.Sequential(nn.Linear(text_dim, hidden), nn.ReLU(), nn.Dropout(0.35))
            self.head = nn.Sequential(nn.Linear(hidden * 4, hidden), nn.ReLU(), nn.Dropout(0.45), nn.Linear(hidden, 2))
        def forward(self, text, image):
            t = self.text(text)
            i = self.image(image)
            return self.head(torch.cat([t, i, torch.abs(t - i), t * i], dim=1))

    class FrozenResNet50FusionNet(nn.Module):
        def __init__(self, text_dim, hidden=128):
            super().__init__()
            if not TORCHVISION_OK:
                raise RuntimeError('Torchvision unavailable.')
            weights = models.ResNet50_Weights.DEFAULT
            backbone = models.resnet50(weights=weights)
            feat_dim = backbone.fc.in_features
            backbone.fc = nn.Identity()
            for p in backbone.parameters():
                p.requires_grad = False
            self.backbone = backbone
            self.text = nn.Sequential(nn.Linear(text_dim, hidden), nn.ReLU(), nn.Dropout(0.35))
            self.image_proj = nn.Sequential(nn.Linear(feat_dim, hidden), nn.ReLU())
            self.head = nn.Sequential(nn.Linear(hidden * 4, hidden), nn.ReLU(), nn.Dropout(0.45), nn.Linear(hidden, 2))
        def forward(self, text, image):
            with torch.no_grad():
                raw_i = self.backbone(image)
            t = self.text(text)
            i = self.image_proj(raw_i)
            return self.head(torch.cat([t, i, torch.abs(t - i), t * i], dim=1))

    def evaluate_torch_model(model, loader):
        model.eval()
        ys, preds, probs = [], [], []
        with torch.no_grad():
            for text, image, y in loader:
                text, image = text.to(DEVICE), image.to(DEVICE)
                logits = model(text, image)
                prob = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
                pred = logits.argmax(dim=1).detach().cpu().numpy()
                ys.extend(y.numpy().tolist())
                preds.extend(pred.tolist())
                probs.extend(prob.tolist())
        return np.asarray(ys), np.asarray(preds), np.asarray(probs)

    def train_model(model, train_loader, val_loader, seed, model_name, lr=2e-4, weight_decay=1e-4):
        set_seed(seed)
        model = model.to(DEVICE)
        y_train_local = train_df['label_id'].astype(int).values
        counts = np.bincount(y_train_local, minlength=2)
        weights = torch.tensor(len(y_train_local) / (2 * np.maximum(counts, 1)), dtype=torch.float32, device=DEVICE)
        criterion = nn.CrossEntropyLoss(weight=weights)
        optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=weight_decay)
        best_state, best_f1, bad_epochs = None, -1, 0
        history = []
        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            total_loss = 0.0
            for text, image, y in train_loader:
                text, image, y = text.to(DEVICE), image.to(DEVICE), y.to(DEVICE)
                optimizer.zero_grad()
                loss = criterion(model(text, image), y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                total_loss += float(loss.item()) * len(y)
            yv, pv, qv = evaluate_torch_model(model, val_loader)
            val_f1 = f1_score(yv, pv, average='macro', zero_division=0)
            history.append({'epoch': epoch, 'train_loss': total_loss / len(train_loader.dataset), 'val_macro_f1': val_f1})
            if val_f1 > best_f1 + MIN_DELTA:
                best_f1 = val_f1
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                bad_epochs = 0
            else:
                bad_epochs += 1
            if bad_epochs >= PATIENCE:
                break
        if best_state is not None:
            model.load_state_dict(best_state)
        return model, pd.DataFrame(history)

    train_loader = DataLoader(FilteredMultimodalDataset(train_df, X_train_text, transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(FilteredMultimodalDataset(val_df, X_val_text, transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(FilteredMultimodalDataset(test_df, X_test_text, transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model_specs = [
        ('Filtered GT6 Image-only CNN', lambda: ImageOnlyNet()),
        ('Filtered GT6 Concat fusion CNN+Text', lambda: ConcatFusionNet(X_train_text.shape[1])),
        ('Filtered GT6 Consistency fusion CNN+Text', lambda: ConsistencyFusionNet(X_train_text.shape[1])),
    ]
    if RUN_FROZEN_RESNET50:
        model_specs.append(('Filtered GT6 Frozen ResNet50 fusion CNN+Text', lambda: FrozenResNet50FusionNet(X_train_text.shape[1])))

    deep_rows = []
    for seed in RETRAIN_SEEDS:
        for name, ctor in model_specs:
            print(f'\\nTraining {name} seed={seed}')
            try:
                model, hist = train_model(ctor(), train_loader, val_loader, seed, name)
                y_true, y_pred, prob_fake = evaluate_torch_model(model, test_loader)
                run_name = f'{name} seed{seed}'
                deep_rows.append({'seed': seed, **compute_metrics(y_true, y_pred, prob_fake, run_name)})
                hist.to_csv(EXPERIMENT_DIR / f'{re.sub(r"[^A-Za-z0-9_.+-]+", "_", run_name)}_history.csv', index=False)
                pred_df, pred_path = save_prediction_file(test_df, y_pred, prob_fake, re.sub(r"[^A-Za-z0-9_.+-]+", "_", run_name))
                prediction_frames.append(pred_df.assign(model_name=run_name, model_family=name))
                torch.save(model.state_dict(), EXPERIMENT_DIR / f'{re.sub(r"[^A-Za-z0-9_.+-]+", "_", run_name)}.pt')
                display(pd.DataFrame([deep_rows[-1]]))
            except Exception as exc:
                print(f'Skipped {name} seed={seed}: {repr(exc)}')
                deep_rows.append({'seed': seed, 'model': f'{name} seed{seed}', 'status': 'skipped', 'reason': repr(exc)})

    deep_results = pd.DataFrame(deep_rows)
    deep_results.to_csv(EXPERIMENT_DIR / 'filtered_gt6_deep_retrain_metrics.csv', index=False)
    display(deep_results)

Device: cuda
\nTraining Filtered GT6 Image-only CNN seed=42


## 6. Optional BERT Text Retraining

In [ ]:
if RUN_TEXT_BERT:
    try:
        import torch
        from torch.utils.data import Dataset, DataLoader
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        import torch.nn as nn
        import torch.optim as optim

        DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        BERT_MODEL_NAME = 'bert-base-uncased'
        BERT_EPOCHS = 4
        BERT_BATCH_SIZE = 16
        BERT_LR = 2e-5

        tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME, local_files_only=True)

        class TextDataset(Dataset):
            def __init__(self, texts, labels):
                self.texts = list(texts)
                self.labels = np.asarray(labels, dtype=int)
            def __len__(self):
                return len(self.labels)
            def __getitem__(self, idx):
                enc = tokenizer(self.texts[idx], truncation=True, padding='max_length', max_length=128, return_tensors='pt')
                return {k: v.squeeze(0) for k, v in enc.items()}, torch.tensor(self.labels[idx], dtype=torch.long)

        def collate_text(batch):
            keys = batch[0][0].keys()
            enc = {k: torch.stack([b[0][k] for b in batch]) for k in keys}
            y = torch.stack([b[1] for b in batch])
            return enc, y

        train_loader = DataLoader(TextDataset(train_df[TEXT_COL].fillna('').astype(str), train_df['label_id']), batch_size=BERT_BATCH_SIZE, shuffle=True, collate_fn=collate_text)
        val_loader = DataLoader(TextDataset(val_df[TEXT_COL].fillna('').astype(str), val_df['label_id']), batch_size=BERT_BATCH_SIZE, shuffle=False, collate_fn=collate_text)
        test_loader = DataLoader(TextDataset(test_df[TEXT_COL].fillna('').astype(str), test_df['label_id']), batch_size=BERT_BATCH_SIZE, shuffle=False, collate_fn=collate_text)

        model = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL_NAME, num_labels=2, local_files_only=True).to(DEVICE)
        optimizer = optim.AdamW(model.parameters(), lr=BERT_LR, weight_decay=1e-4)
        best_state, best_f1 = None, -1
        bad = 0
        history = []

        def eval_bert(loader):
            model.eval()
            ys, preds, probs = [], [], []
            with torch.no_grad():
                for enc, y in loader:
                    enc = {k: v.to(DEVICE) for k, v in enc.items()}
                    logits = model(**enc).logits
                    prob = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
                    pred = logits.argmax(dim=1).detach().cpu().numpy()
                    ys.extend(y.numpy().tolist())
                    preds.extend(pred.tolist())
                    probs.extend(prob.tolist())
            return np.asarray(ys), np.asarray(preds), np.asarray(probs)

        for epoch in range(1, BERT_EPOCHS + 1):
            model.train()
            total = 0.0
            for enc, y in train_loader:
                enc = {k: v.to(DEVICE) for k, v in enc.items()}
                y = y.to(DEVICE)
                optimizer.zero_grad()
                out = model(**enc, labels=y)
                out.loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                total += float(out.loss.item()) * len(y)
            yv, pv, qv = eval_bert(val_loader)
            vf1 = f1_score(yv, pv, average='macro', zero_division=0)
            history.append({'epoch': epoch, 'train_loss': total / len(train_df), 'val_macro_f1': vf1})
            print(f'BERT epoch {epoch}: val macro-F1={vf1:.4f}')
            if vf1 > best_f1 + MIN_DELTA:
                best_f1 = vf1
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                bad = 0
            else:
                bad += 1
            if bad >= PATIENCE:
                break

        if best_state is not None:
            model.load_state_dict(best_state)
        yt, pt, qt = eval_bert(test_loader)
        bert_metrics = pd.DataFrame([compute_metrics(yt, pt, qt, 'Filtered GT6 BERT text classifier')])
        bert_metrics.to_csv(EXPERIMENT_DIR / 'filtered_gt6_bert_text_metrics.csv', index=False)
        pd.DataFrame(history).to_csv(EXPERIMENT_DIR / 'filtered_gt6_bert_text_history.csv', index=False)
        pred_df, pred_path = save_prediction_file(test_df, pt, qt, 'filtered_gt6_bert_text')
        prediction_frames.append(pred_df.assign(model_name='filtered_gt6_bert_text', model_family='text_bert'))
        display(bert_metrics)
    except Exception as exc:
        print('BERT retraining skipped:', repr(exc))
        print('If needed, cache bert-base-uncased locally or set RUN_TEXT_BERT=False.')

## 7. Optional Rich Cached Feature MLP Retraining

In [ ]:
if RUN_RICH_CACHED_MLP:
    try:
        import torch
        import torch.nn as nn
        import torch.optim as optim
        from torch.utils.data import TensorDataset, DataLoader
        from sklearn.preprocessing import StandardScaler

        DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        def load_rich_filtered(split, keep_ids):
            meta = pd.read_csv(FINAL_ARTIFACT_DIR / f'rich_multimodal_{split}_ocr_blip_text.csv')
            z = np.load(FINAL_ARTIFACT_DIR / f'rich_multimodal_{split}_features.npz')
            mask = meta['id'].astype(str).isin(set(map(str, keep_ids))).values
            return meta.loc[mask].reset_index(drop=True), z['features'][mask].astype(np.float32), z['labels'][mask].astype(int)

        train_meta, Xtr, ytr = load_rich_filtered('train', train_df['id'])
        val_meta, Xva, yva = load_rich_filtered('val', val_df['id'])
        test_meta, Xte, yte = load_rich_filtered('test', test_df['id'])

        scaler = StandardScaler()
        Xtr = scaler.fit_transform(Xtr).astype(np.float32)
        Xva = scaler.transform(Xva).astype(np.float32)
        Xte = scaler.transform(Xte).astype(np.float32)

        class RichMLP(nn.Module):
            def __init__(self, dim):
                super().__init__()
                self.net = nn.Sequential(
                    nn.Linear(dim, 512), nn.ReLU(), nn.BatchNorm1d(512), nn.Dropout(0.45),
                    nn.Linear(512, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.35),
                    nn.Linear(128, 2),
                )
            def forward(self, x):
                return self.net(x)

        train_loader = DataLoader(TensorDataset(torch.tensor(Xtr), torch.tensor(ytr, dtype=torch.long)), batch_size=64, shuffle=True)
        val_loader = DataLoader(TensorDataset(torch.tensor(Xva), torch.tensor(yva, dtype=torch.long)), batch_size=128, shuffle=False)
        test_loader = DataLoader(TensorDataset(torch.tensor(Xte), torch.tensor(yte, dtype=torch.long)), batch_size=128, shuffle=False)

        model = RichMLP(Xtr.shape[1]).to(DEVICE)
        weights = torch.tensor(len(ytr) / (2 * np.maximum(np.bincount(ytr, minlength=2), 1)), dtype=torch.float32, device=DEVICE)
        criterion = nn.CrossEntropyLoss(weight=weights)
        optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
        best_state, best_f1, bad = None, -1, 0
        history = []

        def eval_mlp(loader):
            model.eval()
            ys, preds, probs = [], [], []
            with torch.no_grad():
                for xb, yb in loader:
                    xb = xb.to(DEVICE)
                    logits = model(xb)
                    prob = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
                    pred = logits.argmax(dim=1).cpu().numpy()
                    ys.extend(yb.numpy().tolist())
                    preds.extend(pred.tolist())
                    probs.extend(prob.tolist())
            return np.asarray(ys), np.asarray(preds), np.asarray(probs)

        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            total = 0.0
            for xb, yb in train_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                optimizer.zero_grad()
                loss = criterion(model(xb), yb)
                loss.backward()
                optimizer.step()
                total += float(loss.item()) * len(yb)
            yv, pv, qv = eval_mlp(val_loader)
            vf1 = f1_score(yv, pv, average='macro', zero_division=0)
            history.append({'epoch': epoch, 'train_loss': total / len(ytr), 'val_macro_f1': vf1})
            if vf1 > best_f1 + MIN_DELTA:
                best_f1 = vf1
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                bad = 0
            else:
                bad += 1
            if bad >= PATIENCE:
                break

        if best_state is not None:
            model.load_state_dict(best_state)
        yt, pt, qt = eval_mlp(test_loader)
        rich_metrics = pd.DataFrame([compute_metrics(yt, pt, qt, 'Filtered GT6 OCR+BLIP+BERT+ResNet50 cached-feature MLP')])
        rich_metrics.to_csv(EXPERIMENT_DIR / 'filtered_gt6_rich_cached_mlp_metrics.csv', index=False)
        pd.DataFrame(history).to_csv(EXPERIMENT_DIR / 'filtered_gt6_rich_cached_mlp_history.csv', index=False)

        # test_meta order matches Xte/yte.
        test_meta2 = test_meta.copy()
        if 'label_id' not in test_meta2.columns:
            test_meta2['label_id'] = test_meta2['label'].map(LABEL_TO_ID)
        pred_df, pred_path = save_prediction_file(test_meta2, pt, qt, 'filtered_gt6_rich_cached_mlp')
        prediction_frames.append(pred_df.assign(model_name='filtered_gt6_rich_cached_mlp', model_family='rich_multimodal'))
        display(rich_metrics)
    except Exception as exc:
        print('Rich cached-feature MLP retraining skipped:', repr(exc))
        print('Run the rich branch once first if cached rich_multimodal_* feature files are missing.')

## 8. Voting Ensemble and Original-vs-Filtered Comparison

In [ ]:
all_metric_files = [
    EXPERIMENT_DIR / 'filtered_gt6_text_baseline_metrics.csv',
    EXPERIMENT_DIR / 'filtered_gt6_deep_retrain_metrics.csv',
    EXPERIMENT_DIR / 'filtered_gt6_bert_text_metrics.csv',
    EXPERIMENT_DIR / 'filtered_gt6_rich_cached_mlp_metrics.csv',
]

metric_frames = []
for path in all_metric_files:
    if path.exists():
        df = pd.read_csv(path)
        df['metric_file'] = path.name
        metric_frames.append(df)

filtered_metrics = pd.concat(metric_frames, ignore_index=True, sort=False) if metric_frames else pd.DataFrame()

if RUN_VOTING_ENSEMBLE and prediction_frames:
    votes = pd.concat(prediction_frames, ignore_index=True)
    vote_rows = []
    for sample_id, group in votes.groupby('id'):
        y = int(group['label_id'].iloc[0])
        fake_vote_rate = float(group['pred_id'].astype(int).mean())
        avg_prob = pd.to_numeric(group['prob_fake'], errors='coerce').mean()
        vote_rows.append({
            'id': sample_id,
            'label_id': y,
            'label': ID_TO_LABEL[y],
            'hard_vote_pred_id': int(fake_vote_rate >= 0.5),
            'soft_vote_pred_id': int(avg_prob >= 0.5),
            'fake_vote_rate': fake_vote_rate,
            'avg_prob_fake': avg_prob,
            'n_model_runs': int(group['model_name'].nunique()),
            'n_model_families': int(group['model_family'].nunique()),
        })
    vote_pred = pd.DataFrame(vote_rows)
    vote_metrics = pd.DataFrame([
        compute_metrics(vote_pred['label_id'], vote_pred['hard_vote_pred_id'], vote_pred['fake_vote_rate'], 'Filtered GT6 all-retrained hard vote'),
        compute_metrics(vote_pred['label_id'], vote_pred['soft_vote_pred_id'], vote_pred['avg_prob_fake'], 'Filtered GT6 all-retrained soft vote'),
    ])
    vote_pred.to_csv(EXPERIMENT_DIR / 'filtered_gt6_voting_predictions.csv', index=False)
    vote_metrics.to_csv(EXPERIMENT_DIR / 'filtered_gt6_voting_metrics.csv', index=False)
    filtered_metrics = pd.concat([filtered_metrics, vote_metrics.assign(metric_file='filtered_gt6_voting_metrics.csv')], ignore_index=True, sort=False)

if len(filtered_metrics):
    filtered_metrics.to_csv(EXPERIMENT_DIR / 'FILTERED_GT6_ALL_RETRAINED_METRICS.csv', index=False)
    display(filtered_metrics[['model', 'n_samples', 'accuracy', 'macro_f1', 'roc_auc', 'pr_auc', 'metric_file']].sort_values(['macro_f1', 'accuracy'], ascending=False))
else:
    print('No filtered metrics available yet. Run the training sections above.')

original_path = FINAL_ARTIFACT_DIR / 'report_table_classification_results.csv'
if original_path.exists() and len(filtered_metrics):
    original = pd.read_csv(original_path)
    original['experiment'] = 'original_main_test'
    filtered_show = filtered_metrics.copy()
    filtered_show['experiment'] = 'filtered_gt6_retrained'
    compare = pd.concat([
        original[['experiment', 'model', 'accuracy', 'macro_f1', 'roc_auc', 'pr_auc']] if set(['model','accuracy','macro_f1']).issubset(original.columns) else pd.DataFrame(),
        filtered_show[['experiment', 'model', 'accuracy', 'macro_f1', 'roc_auc', 'pr_auc']]
    ], ignore_index=True, sort=False)
    compare.to_csv(EXPERIMENT_DIR / 'original_vs_filtered_gt6_retrained_comparison.csv', index=False)
    display(compare)

print('Saved filtered experiment artifacts under:', EXPERIMENT_DIR)